# Part 2: Inverse Kinematics

Jacobian based IK, following the course pseudocode (pg 14-7).
- Rotation error: **k/θ** (`s = kθ`, `eR = θ`)
- Update: **damped least squares**

Keep this notebook in the same folder as `DofbotKinematics.ipynb`.

In [ ]:
%run DofbotKinematics.ipynb

import time
from Arm_Lib import Arm_Device

arm = Arm_Device()

## Jacobian J(q)
Each column is one joint. Top 3 rows: how fast the gripper rotates (the joint axis `h`).
Bottom 3 rows: how fast the tip moves (`h × (tip - joint location)`).
Units are per radian. Uses the same axes and link vectors as `fk_Dofbot`.

In [ ]:
def jacobian(q):
    # Rotation from the base up to each joint
    R01 = rotz(q[0])
    R02 = R01 * roty(-q[1])
    R03 = R02 * roty(-q[2])
    R04 = R03 * roty(-q[3])
    R05 = R04 * rotx(-q[4])

    # Location of each joint and the tip, in the base frame
    p1 = np.zeros(3)
    p2 = (l0 + l1) * ez
    p3 = p2 + R02.apply(l2 * ex)
    p4 = p3 + R03.apply(-l3 * ez)
    p5 = p4
    pT = p5 + R05.apply(-(l4 + l5) * ex)

    # Joint axes in the base frame
    h = [R01.apply(ez), R02.apply(-ey), R03.apply(-ey), R04.apply(-ey), R05.apply(-ex)]
    p = [p1, p2, p3, p4, p5]

    J = np.zeros((6, 5))
    for i in range(5):
        J[:3, i] = h[i]                          # rotation part
        J[3:, i] = np.cross(h[i], pT - p[i])     # position part
    return J

## IK function (the pseudocode)
| Pseudocode | Code |
|---|---|
| `q0, Rd, Pd` | inputs |
| `P = fp(q0), R = fR(q0)` | `R, P = fk_Dofbot(q)` |
| `ep = ½‖P − Pd‖²` | `ep` |
| `ER = R Rdᵀ` | `ER = R * Rd.inv()` |
| convert ER to k/θ | `s = ER.as_rotvec()` (this is kθ) |
| `eR` | `eR = θ` |
| `J = J(q)` | `jacobian(q)` |
| `Uq` = damped least squares `[βs; P − Pd]` | `Jᵀ(JJᵀ + λI)⁻¹ [βs; P − Pd]` |
| `q = q − αUq` | same, then kept inside the servo range 0 to 180 |

`alpha` = step size, `beta` = how much orientation matters vs position, `lam` = damping (keeps steps small near singular poses).
`max_iter` stops the loop for poses the arm can't reach.

In [ ]:
def ik_Dofbot(Rd, Pd, q0, alpha=0.3, beta=1.0, lam=0.001, tol_p=1e-7, tol_R=1e-3, max_iter=1000):
    # Initialize
    q = np.array(q0, dtype=float)
    R, P = fk_Dofbot(q)
    ep = 0.5 * np.linalg.norm(P - Pd)**2      # position error
    ER = R * Rd.inv()                         # rotation error matrix
    s = ER.as_rotvec()                        # k*theta
    eR = np.linalg.norm(s)                    # theta
    J = jacobian(q)

    it = 0
    while (eR > tol_R or ep > tol_p) and it < max_iter:
        # Damped least squares update
        Uq = J.T @ np.linalg.inv(J @ J.T + lam * np.eye(6)) @ np.concatenate([beta * s, P - Pd])
        q = np.clip(q - alpha * np.degrees(Uq), 0, 180)

        # Recompute everything at the new q
        J = jacobian(q)
        R, P = fk_Dofbot(q)
        ER = R * Rd.inv()
        ep = 0.5 * np.linalg.norm(P - Pd)**2
        s = ER.as_rotvec()
        eR = np.linalg.norm(s)
        it += 1

    return q

## Five desired end effector poses
Each desired pose `(Rd, Pd)` is made with the FK from a set of angles, so we know the arm can reach it.
The IK never sees those angles. It only gets `Rd` and `Pd`.

Initial guess `q0`: a slightly bent arm. Don't use all 90s, because straight up is a singular pose.

In [ ]:
q0 = [90, 110, 80, 70, 90]

source_angles = [
    [90, 150, 30, 120, 90],
    [45, 130, 60, 90, 90],
    [120, 140, 50, 80, 45],
    [135, 130, 60, 130, 90],
    [30, 140, 30, 150, 60],
]

desired_poses = [fk_Dofbot(q) for q in source_angles]   # list of (Rd, Pd)

for Rd, Pd in desired_poses:
    print("Pd (m):", np.round(Pd, 4))

## Run IK, command the arm, measure, compare
For each pose: solve IK, send the angles to the arm, then type in the tip position you measured (meters, `x y z`).
Only position is measured.

In [ ]:
for Rd, Pd in desired_poses:
    q = ik_Dofbot(Rd, Pd, q0)
    q_cmd = [int(round(a)) for a in q]

    print("\nDesired position Pd (m):", np.round(Pd, 4))
    print("IK joint angles (deg):", np.round(q, 1))

    arm.Arm_serial_servo_write6(*q_cmd, 90, 2000)
    time.sleep(2.5)

    m = input("Measured x y z in meters (space separated): ")
    measured = np.array([float(v) for v in m.split()])

    print("Position error (m):", np.round(measured - Pd, 4))
    print("Error distance (m):", round(np.linalg.norm(measured - Pd), 4))

# 6000 Level: Workspace Exploration

Three poses the arm can't reach:
1. **Too far:** the position is past the arm's reach (about 0.30 m from the shoulder).
2. **Too high:** the position is above the arm's max height (0.399 m, arm straight up).
3. **Impossible orientation:** a reachable position, but the gripper is turned 90° out of the arm's bending plane. The arm only has 5 joints, so it can't do that.

**Nearby metric:** the IK's own error, `‖[β·kθ ; P − Pd]‖`. Damped least squares keeps lowering this error. When the pose is unreachable, the loop can't reach zero, so it ends at the reachable pose with the smallest error, staying inside the servo range. That final pose is our nearby achievable pose. We report its position distance `‖P − Pd‖` and orientation angle `θ` separately.

**β picks what to relax.** A large β (1.0) keeps orientation and relaxes position. A small β (0.01) keeps position and relaxes orientation.

In [ ]:
# (name, Rd, Pd, beta)
unreachable = [
    ("Too far",           fk_Dofbot([90, 150, 30, 120, 90])[0],             np.array([0, -0.30, 0.30]), 1.0),
    ("Too high",          fk_Dofbot([90, 90, 90, 90, 90])[0],               np.array([0, 0, 0.45]),     1.0),
    ("Impossible orient", rotz(90) * fk_Dofbot([45, 130, 60, 90, 90])[0],   fk_Dofbot([45, 130, 60, 90, 90])[1], 0.01),
]

for name, Rd, Pd, beta in unreachable:
    q = ik_Dofbot(Rd, Pd, q0, beta=beta)
    R, P = fk_Dofbot(q)
    theta = np.degrees(np.linalg.norm((R * Rd.inv()).as_rotvec()))

    print(f"\n{name}  (beta = {beta})")
    print("Desired position (m):  ", np.round(Pd, 4))
    print("Achievable position (m):", np.round(P, 4))
    print("Nearby joint angles (deg):", np.round(q, 1))
    print("Position distance (m):", round(np.linalg.norm(P - Pd), 4))
    print("Orientation angle (deg):", round(theta, 1))